In [3]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from feature_engineer import get_engineered_df

In [4]:
# 1. Load and Preprocess once
df, features, cat_cols = get_engineered_df("../data/processed/oe_detailed.parquet", warehouse="OE", work_code="30")
X = pd.get_dummies(df[features], columns=cat_cols, drop_first=True)
y = df["Time_Delta_sec"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# 2. Define Objectives to test
objectives = [
    "reg:squarederror",
    "reg:squaredlogerror",
    "reg:gamma",
    "reg:tweedie",
    "reg:absoluteerror"
]

obj_results = []

In [8]:
for obj in objectives:
    print(f"Testing objective: {obj}")
    
    # Initialize model with specific objective
    # Note: tree_method='hist' is recommended for faster training with these objectives
    # add runtime recording
    model = XGBRegressor(
        objective=obj,
        n_estimators=1000,
        learning_rate=0.05,
        tree_method="hist", 
        random_state=42
    )
    
    time_start = pd.Timestamp.now()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    time_end = pd.Timestamp.now()
    runtime = (time_end - time_start).total_seconds()
    
    obj_results.append({
        "Objective": obj,
        "R^2": r2_score(y_test, preds),
        "MAE": mean_absolute_error(y_test, preds),
        "Runtime": runtime
    })

# Comparison Table
comparison_df = pd.DataFrame(obj_results).sort_values("R^2", ascending=False)
display(comparison_df)

Testing objective: reg:squarederror
Testing objective: reg:squaredlogerror
Testing objective: reg:gamma
Testing objective: reg:tweedie
Testing objective: reg:absoluteerror


,Objective,R^2,MAE,Runtime
0,reg:squarederror,0.390775,22.345789,NaN
5,reg:squarederror,0.390775,22.345789,0.800908
3,reg:tweedie,0.387456,22.064574,NaN
8,reg:tweedie,0.387456,22.064574,0.941282
2,reg:gamma,0.386159,21.912330,NaN
7,reg:gamma,0.386159,21.912330,0.828913
4,reg:absoluteerror,0.340829,21.177365,NaN
9,reg:absoluteerror,0.340829,21.177365,5.963535
1,reg:squaredlogerror,0.259914,22.679350,NaN
6,reg:squaredlogerror,0.259914,22.679350,0.835838
